# 03. CSV

**Tier:** Intermediate. **Why you'd use this:** A spreadsheet export is comma-separated text. You need rows as dicts and correct line handling so Excel-style files do not invent blank rows.

**After this lesson you can:** read rows with `csv.DictReader`; sum a column; open CSV files with `newline=""`.
About 25 minutes. **Uses:** [Paths](01-paths-lesson.ipynb), [JSON](02-json-lesson.ipynb)


## Quick review

1. What happens to a tuple through JSON dump/load?
2. Why are JSON object keys strings?

<details><summary>Answers</summary>

It becomes a list. The JSON format only has string keys.
</details>


## The idea

`csv.DictReader` turns the header row into dict keys for each later row. Values arrive as strings, so convert numbers yourself. When opening a CSV file, pass `newline=""` so the csv module owns line ending translation.

<details><summary>Explain more</summary>

`io.StringIO` lets you practice on a text blob without a real file. File opens still want `encoding` and `newline=""`.
</details>



## See it work

Sum quantities from a CSV string.


In [1]:
import csv
import io

text = "name,qty\nada,2\nbea,3\n"
# 1. DictReader uses the header for keys.
rows = csv.DictReader(io.StringIO(text))
print(sum(int(row["qty"]) for row in rows))


5


## Check yourself

Answer these before you run the cell.

1. What type is `row["qty"]` before `int`?
2. Why open CSV files with `newline=""`?


In [2]:
import csv
import io

rows = list(csv.DictReader(io.StringIO("name,qty\nada,2\n")))
print(rows[0]["qty"], type(rows[0]["qty"]).__name__)


2 str


<details><summary>Answers</summary>

`str`. So the csv module controls newlines and does not mis-count rows on Excel-style files.
</details>


## Why it works this way

CSV is rows of fields, not a typed document format. The csv module parses delimiters and quotes. `DictReader` consumes the first row as field names and yields dicts. Every cell is a string until you convert it. That keeps parsing separate from typing. Deep dive: same boundary idea as JSON, with weaker types.

`newline=""` on open is required by the csv docs so the module can handle universal newlines itself. Without it, some files produce extra blank rows or break quoted multiline fields on Windows. Pair it with `encoding="utf-8"` for text CSVs.

Cost is one pass over the rows. For huge files, iterate instead of `list(...)`. Prefer DictReader when columns have names. Prefer reader when there is no header.

Evidence: `qty` is a string until `int`, and a short StringIO example sums correctly. When you write CSV, use `DictWriter` with the same `newline=""` rule.  Dialects exist for rare delimiter rules. Start with the defaults and DictReader before you customize. Round-trip a file in tests with the same newline and encoding settings you use in production.

Quoted fields can contain commas and newlines. Hand-split code breaks on the first real spreadsheet export. The csv module exists to parse those rules. When writing, always write the header if you use DictReader on the other side. Keep one canonical column order in writers so diffs stay readable in version control.  Empty files and header-only files are normal. Decide whether zero rows is an error or a valid total before you write the reader.

That cost shows up when the input grows: each extra unit of work multiplies wall time or memory, so the design choice is not only correctness but whether the approach stays practical under load.


In [3]:
import csv
import io
from pathlib import Path

text = "name,qty\nada,2\nbea,3\n"
print(sum(int(row["qty"]) for row in csv.DictReader(io.StringIO(text))))
path = Path("items.csv")
with path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["name", "qty"])
    writer.writeheader()
    writer.writerow({"name": "ada", "qty": 2})
with path.open(encoding="utf-8", newline="") as handle:
    print(list(csv.DictReader(handle)))
path.unlink()


5
[{'name': 'ada', 'qty': '2'}]


## When to use it

Use csv for tabular text exchanges. Convert cell types explicitly. Always open with `newline=""` and an encoding. Don't split rows on commas by hand when quoting matters. Prefer one writer helper so every export shares newline and encoding settings.


## Common mistake

Opening a CSV file without `newline=""` and getting blank rows or broken quotes on some platforms.


In [4]:
import csv
from pathlib import Path

path = Path("items.csv")
path.write_text("name,qty\rada,2\n", encoding="utf-8")
with path.open(encoding="utf-8") as handle:
    print(list(csv.DictReader(handle)))
path.unlink()


[{'name': 'ada', 'qty': '2'}]


Pass `newline=""` so the csv module owns line endings.


In [5]:
import csv
from pathlib import Path

path = Path("items.csv")
with path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["name", "qty"])
    writer.writeheader()
    writer.writerow({"name": "ada", "qty": 2})
with path.open(encoding="utf-8", newline="") as handle:
    print(list(csv.DictReader(handle)))
path.unlink()


[{'name': 'ada', 'qty': '2'}]


## Try it

Read a CSV with a missing qty treated as 0. Where do you put the conversion, and what happens if you forget `int`?


## Recap

- `DictReader` yields dict rows with string values.
- Convert numbers yourself.
- Open CSV files with `newline=""` and an encoding.

**Next:** [practice](03-csv-practice.ipynb)